<a href="https://colab.research.google.com/github/Salma-221/Calculator-Number-Systems-Convertor/blob/master/GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile app.py

import streamlit as st
import numpy as np
import cv2
from PIL import Image, ImageDraw
import tensorflow as tf
import random

# ============================
#        FILTER FUNCTIONS
# ============================

# ------ Sketch Filter ------
def gaussian_kernel(size=15, sigma=3):
    x = tf.range(-size // 2 + 1, size // 2 + 1, dtype=tf.float32)
    x = tf.math.exp(-0.5 * (x / sigma) ** 2)
    g = tf.tensordot(x, x, axes=0)
    g /= tf.reduce_sum(g)
    return g

def grayscale_invert_and_sketch(image):
    img = tf.image.convert_image_dtype(image, tf.float32)
    gray_img = tf.image.rgb_to_grayscale(img)
    inverted_img = 1.0 - gray_img

    g_kernel = gaussian_kernel(15, 3)
    g_kernel = g_kernel[:, :, tf.newaxis, tf.newaxis]
    blurred = tf.nn.conv2d(inverted_img[tf.newaxis, ...], g_kernel,
                           strides=[1,1,1,1], padding='SAME')
    blurred = tf.squeeze(blurred, 0)

    sketch = gray_img / (1.0 - blurred + 1e-6)
    sketch = tf.clip_by_value(sketch, 0.0, 1.0)

    sketch = (sketch.numpy() * 255).astype(np.uint8)
    sketch = np.repeat(sketch, 3, axis=2)
    return sketch

# ------ Oil Painting ------
def oil_painting(image, radius=4, intensity_levels=25):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    output = np.zeros_like(image)
    h, w = gray.shape

    for y in range(h):
        for x in range(w):
            intensity_count = np.zeros(intensity_levels)
            b_sum = np.zeros(intensity_levels)
            g_sum = np.zeros(intensity_levels)
            r_sum = np.zeros(intensity_levels)

            for dy in range(-radius, radius + 1):
                for dx in range(-radius, radius + 1):
                    yy = min(max(y + dy, 0), h - 1)
                    xx = min(max(x + dx, 0), w - 1)

                    intensity = (int(gray[yy, xx]) * intensity_levels) // 256
                    intensity_count[intensity] += 1

                    b, g, r = image[yy, xx]
                    b_sum[intensity] += b
                    g_sum[intensity] += g
                    r_sum[intensity] += r

            dom = np.argmax(intensity_count)
            cnt = max(intensity_count[dom], 1)
            output[y, x] = (b_sum[dom] / cnt,
                            g_sum[dom] / cnt,
                            r_sum[dom] / cnt)

    return output.astype(np.uint8)

# ------ Pixel Art ------
def quantize_color(img, levels=8):
    step = 256 // levels
    return (img // step) * step + step // 2

def pixel_art(image, scale_factor=0.45, color_levels=18):
    small = cv2.resize(image, None, fx=scale_factor, fy=scale_factor,
                       interpolation=cv2.INTER_LINEAR)
    small_quant = quantize_color(small, color_levels)
    pixel = cv2.resize(small_quant, (image.shape[1], image.shape[0]),
                       interpolation=cv2.INTER_NEAREST)
    return pixel

# ------ Cartoon ------
def cartoon_filter(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 7)
    edges = cv2.adaptiveThreshold(gray, 255,
                                  cv2.ADAPTIVE_THRESH_MEAN_C,
                                  cv2.THRESH_BINARY, 9, 2)
    color = cv2.bilateralFilter(image, 9, 300, 300)
    cartoon = cv2.bitwise_and(color, color, mask=edges)
    return cartoon

# ------ Pointillism ------
def pointillism_filter(image, dot_size=3, num_dots=8000):
    h, w, _ = image.shape
    canvas = Image.new("RGB", (w, h), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    for _ in range(num_dots):
        x = random.randint(0, w - 1)
        y = random.randint(0, h - 1)
        r = image[y, x]
        color = (int(r[0]), int(r[1]), int(r[2]))
        draw.ellipse((x - dot_size, y - dot_size, x + dot_size, y + dot_size),
                     fill=color)

    return np.array(canvas)

# ------ Blur ------
def blur_filter(image):
    return cv2.GaussianBlur(image, (21, 21), 0)

# ============================
#        STREAMLIT APP
# ============================

st.title("🎨 Image Filter App")

uploaded = st.file_uploader("Upload an image", type=["jpg", "png", "jpeg"])

filters = ["Pixel Art", "Sketch", "Oil Painting", "Cartoon", "Pointillism", "Blur"]
choice = st.selectbox("Choose a filter", filters)

if uploaded:
    img = Image.open(uploaded).convert("RGB")
    img_np = np.array(img)

    st.image(img_np, caption="Original Image", width=350)

    if st.button("Apply Filter"):
        with st.spinner("Applying filter..."):

            if choice == "Sketch":
                result = grayscale_invert_and_sketch(img_np)

            elif choice == "Oil Painting":
                bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
                result = oil_painting(bgr)
                result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

            elif choice == "Pixel Art":
                bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
                result = pixel_art(bgr)
                result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

            elif choice == "Cartoon":
                bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
                result = cartoon_filter(bgr)
                result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

            elif choice == "Pointillism":
                result = pointillism_filter(img_np)

            elif choice == "Blur":
                result = blur_filter(img_np)

        st.image(result, caption=f"{choice} Result", width=350)


In [11]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 57.2 MB/s eta 0:00:00


In [14]:
!pip install pyngrok


In [15]:
from pyngrok import ngrok

ngrok.set_auth_token("36ZLHSgPwloVdvw40nDbQ0TVF4h_6NCSHTmhjufcCb3XdXFbF")


In [16]:
public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://nonrepressibly-unfatty-pamelia.ngrok-free.dev" -> "http://localhost:8501">

In [ ]:
!streamlit run app.py --server.address 0.0.0.0 --server.port=8501





  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

http://localhost:8501/
http://localhost:8501/
2025-12-08 16:12:24.490738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765210344.533762   22694 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765210344.546184   22694 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765210344.588622   22694 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765210344.588683   22694 computation_placer.cc:177] computation placer already registered. Please check linkage and 